# QueleaGuard — Ngao Presentation Model Training

## Purpose

This notebook develops the temporary machine-learning models used for the Ngao Labs presentation.

The models are trained from the validated Ngao Presentation dataset and are intended for **presentation/demo purposes**, not as the final production QueleaGuard prediction system.

### Models

- Logistic Regression
- Random Forest

### Validation principle

Repeated environmental feature profiles are kept together during train/test splitting to prevent identical profiles from appearing in both partitions.

### Dataset

`data/processed/ngao_presentation_dataset.csv`

### Output

Trained presentation model artifacts will be saved under:

`models/ngao_presentation/`

> **Important:** These models must not be represented as the final production QueleaGuard models. The production modelling pipeline remains a separate engineering phase.

In [1]:
# Reproducibility and project paths

from pathlib import Path
import sys

import numpy as np
import pandas as pd

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "ngao_presentation_dataset.csv"
MODEL_DIR = PROJECT_ROOT / "models" / "ngao_presentation"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset: {DATA_PATH}")
print(f"Model output directory: {MODEL_DIR}")

Project root: C:\Users\pc\Projects\QueleaGuard
Dataset: C:\Users\pc\Projects\QueleaGuard\data\processed\ngao_presentation_dataset.csv
Model output directory: C:\Users\pc\Projects\QueleaGuard\models\ngao_presentation


In [2]:
# Load the Ngao Presentation dataset

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (259, 15)

Columns:
['rainfall_7d', 'rainfall_30d', 'rainfall_90d', 'temp_mean_7d', 'dewpoint_mean_7d', 'wind_mean_7d', 'temp_same_day', 'dewpoint_same_day', 'wind_same_day', 'ndvi_nearest_composite', 'ndvi_anomaly', 'elevation_m', 'slope_deg', 'dist_to_water_m', 'presence']


In [3]:
# Inspect the target variable

print("Target value counts:")
print(df["presence"].value_counts(dropna=False))

print("\nTarget proportions:")
print(df["presence"].value_counts(normalize=True, dropna=False))

Target value counts:
presence
1    133
0    126
Name: count, dtype: int64

Target proportions:
presence
1    0.513514
0    0.486486
Name: proportion, dtype: float64


In [4]:
# Check for missing values

missing = df.isna().sum()

print("Missing values by column:")
print(missing)

print(f"\nTotal missing values: {missing.sum()}")

Missing values by column:
rainfall_7d               0
rainfall_30d              0
rainfall_90d              0
temp_mean_7d              0
dewpoint_mean_7d          0
wind_mean_7d              0
temp_same_day             0
dewpoint_same_day         0
wind_same_day             0
ndvi_nearest_composite    0
ndvi_anomaly              0
elevation_m               0
slope_deg                 0
dist_to_water_m           0
presence                  0
dtype: int64

Total missing values: 0


In [5]:
# Define features and target

FEATURE_COLUMNS = [
    "rainfall_7d",
    "rainfall_30d",
    "rainfall_90d",
    "temp_mean_7d",
    "dewpoint_mean_7d",
    "wind_mean_7d",
    "temp_same_day",
    "dewpoint_same_day",
    "wind_same_day",
    "ndvi_nearest_composite",
    "ndvi_anomaly",
    "elevation_m",
    "slope_deg",
    "dist_to_water_m",
]

TARGET_COLUMN = "presence"

X = df[FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].copy()

print(f"Number of features: {len(FEATURE_COLUMNS)}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nTarget dtype: {y.dtype}")

Number of features: 14
X shape: (259, 14)
y shape: (259,)

Target dtype: int64


In [6]:
# Identify repeated environmental profiles

profile_counts = (
    df.groupby(FEATURE_COLUMNS, dropna=False)
      .size()
      .sort_values(ascending=False)
)

n_unique_profiles = len(profile_counts)
n_repeated_profiles = (profile_counts > 1).sum()
n_rows_in_repeated_profiles = profile_counts[profile_counts > 1].sum()

print(f"Total rows: {len(df)}")
print(f"Unique environmental profiles: {n_unique_profiles}")
print(f"Profiles occurring more than once: {n_repeated_profiles}")
print(f"Rows belonging to repeated profiles: {n_rows_in_repeated_profiles}")

print("\nMost frequently repeated profiles:")
print(profile_counts.head(10))

Total rows: 259
Unique environmental profiles: 155
Profiles occurring more than once: 60
Rows belonging to repeated profiles: 164

Most frequently repeated profiles:
rainfall_7d  rainfall_30d  rainfall_90d  temp_mean_7d  dewpoint_mean_7d  wind_mean_7d  temp_same_day  dewpoint_same_day  wind_same_day  ndvi_nearest_composite  ndvi_anomaly  elevation_m  slope_deg  dist_to_water_m
0.00         40.92         385.37        26.17         13.26             2.07          28.21          11.19              0.44            0.6030                 -0.0166       1448.0       0.69       1174.5             5
21.68        84.05         636.37        26.06         14.89             1.76          25.99          15.18              2.15            0.0045                 -0.5446       1134.0       2.18       131.6              5
             77.39         625.56        25.69         15.31             1.82          25.31          15.89              1.95            0.0045                 -0.5446       1134.0  

In [7]:
# Check whether repeated environmental profiles have consistent labels

profile_label_counts = (
    df.groupby(FEATURE_COLUMNS)["presence"]
      .nunique()
)

conflicting_profiles = profile_label_counts[profile_label_counts > 1]

print(f"Repeated profiles with conflicting labels: {len(conflicting_profiles)}")

if len(conflicting_profiles) > 0:
    print("\nConflicting profiles:")
    print(conflicting_profiles)
else:
    print("All repeated environmental profiles have a consistent presence label.")

Repeated profiles with conflicting labels: 0
All repeated environmental profiles have a consistent presence label.


In [8]:
# Create validation groups from environmental profiles

groups = (
    df[FEATURE_COLUMNS]
    .astype(str)
    .agg("|".join, axis=1)
)

print(f"Total rows: {len(groups)}")
print(f"Unique validation groups: {groups.nunique()}")
print(f"Groups with multiple rows: {(groups.value_counts() > 1).sum()}")

Total rows: 259
Unique validation groups: 155
Groups with multiple rows: 60


In [ ]:
%pip install scikit-learn